In [1]:
from pathlib import Path
import sys
import os
import numpy as np

PROJECT_ROOT = Path("/afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe")

# Mejor guardar datos pesados fuera del repo.
# Opción A: en tu home NFS del JupyterHub
DATA_ROOT = PROJECT_ROOT/ "data"

# Opción B: en AFS junto al proyecto, si quieres tenerlo todo ahí
# DATA_ROOT = PROJECT_ROOT / "data"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"No existe PROJECT_ROOT: {PROJECT_ROOT}")

DATA_RAW = DATA_ROOT / "raw"
DATA_PROCESSED = DATA_ROOT / "processed"
MODELS_DIR = DATA_ROOT / "models"
RESULTS_DIR = DATA_ROOT / "results"
CHECKPOINTS_DIR = MODELS_DIR / "checkpoints"

for path in [DATA_RAW, DATA_PROCESSED, MODELS_DIR, RESULTS_DIR, CHECKPOINTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("hostname:", os.uname().nodename)
print("python:", sys.executable)
print("cwd:", Path.cwd())
print("DATA_ROOT:", DATA_ROOT)
print("sys.path[:3]:", sys.path[:3])

PROJECT_ROOT: /afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe
PROJECT_ROOT exists: True
hostname: pcae159.ciemat.es
python: /afs/ciemat.es/user/v/vserrano/miniconda3/envs/gw-env/bin/python
cwd: /afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe
DATA_ROOT: /afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe/data
sys.path[:3]: ['/afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe', '/afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe/notebooks', '/opt/utils']


In [ ]:
from pathlib import Path
import numpy as np

from src.io import load_dataset_npz

dataset_id = "bbh_processed_4s_seobnrv4opt_snr10-25_n15000_merged"

dataset_path = DATA_PROCESSED / f"{dataset_id}.npz"
split_paths = sorted(DATA_PROCESSED.glob(f"{dataset_id}_splits*.npz"))

if len(split_paths) == 0:
    raise FileNotFoundError(f"No split files found for dataset_id={dataset_id}")

if len(split_paths) > 1:
    raise ValueError(
        "More than one split file found:\n"
        + "\n".join(str(p) for p in split_paths)
    )

split_path = split_paths[0]

print("Using split file:", split_path)

splits = np.load(split_path)

train_idx = splits["train_idx"]
val_idx = splits["val_idx"]

print(split_path)

batch = load_dataset_npz(dataset_path)

X = batch.X
y = batch.y
metadata = batch.metadata
network_snrs = np.array([ m["snr"]["final_network_snr"] for m in metadata], dtype=np.float32)
label_names = ["chirp_mass", "total_mass", "chi_eff"]

X_train, y_train_phys = X[train_idx], y[train_idx]
X_val, y_val_phys = X[val_idx], y[val_idx]


In [ ]:
label_stats_paths = sorted(DATA_PROCESSED.glob(f"{dataset_id}_label_stats*.npz"))

if len(label_stats_paths) == 0:
    raise FileNotFoundError(f"No label stats found for dataset_id={dataset_id}")

if len(label_stats_paths) > 1:
    raise ValueError(
        "More than one label stats file found:\n"
        + "\n".join(str(p) for p in label_stats_paths)
    )

label_stats_path = label_stats_paths[0]

y_params = np.load(label_stats_path)

y_mean = y_params["y_mean"]
y_std = y_params["y_std"]

y_train_std = (y_train_phys - y_mean) / y_std
y_val_std = (y_val_phys - y_mean) / y_std


## Load the checkpoint

In [ ]:
import torch
from src.models.network import SimpleCNN, SimpleCNN_v2, SimpleCNN_Pool


checkpoint_id = "bbh_processed_4s_seobnrv4opt_snr10-25_n15000_merged_SimpleCNNPool_pool4_MSELoss_emb64_seed123"

checkpoint_file_name = f"{checkpoint_id}_checkpoint.pt"
checkpoint_path = CHECKPOINTS_DIR / checkpoint_file_name

checkpoint = torch.load(checkpoint_path, map_location=device)

model_config = checkpoint["model_config"]

print("Loaded epoch:", checkpoint["epoch"])
print("Loaded best val loss:", checkpoint["best_val_loss"])

In [ ]:
model

## Load the predictions

In [ ]:
#Standardized predictions (y_std where used in the dataloader)
pred_train, y_train, emb_train = predict_set(model, train_loader, device)
pred_val, y_val, emb_val = predict_set(model, val_loader, device)





    """
    El coeficiente R2 mide qué fracción de la varianza del target explica el modelo.
        - R² = 1 -> predicción perfecta
        - R² = 0 -> igual que predecir siempre la media del target
        - R² < 0 -> peor que predecir siempre la media
    """
  

In [ ]:
import pandas as pd
from src.models.evaluate import regression_metrics

metrics_train_std = regression_metrics(y_train, pred_train, label_names, "train_std")
metrics_val_std   = regression_metrics(y_val, pred_val, label_names, "val_std")

metrics_all_std = pd.concat(
    [metrics_train_std, metrics_val_std],
    ignore_index=True
)

metrics_all_std

In physical space

In [ ]:
y_val_phys = y_val * y_std + y_mean
pred_val_phys = pred_val * y_std + y_mean

In [ ]:
metrics_val_phys = regression_metrics(
    y_true=y_val_phys,
    y_pred=pred_val_phys,
    label_names=label_names,
    split_name="test_phys"
)

metrics_val_phys

## 1. Compute some metrics

In [ ]:
from src.models.evaluate import inverse_standardize

pred_train_phys = inverse_standardize(pred_train, y_mean, y_std)
pred_val_phys   = inverse_standardize(pred_val,   y_mean, y_std)

y_train_phys = inverse_standardize(y_train, y_mean, y_std)
y_val_phys   = inverse_standardize(y_val,   y_mean, y_std)


In [ ]:
metrics_train_phys = regression_metrics(y_train_phys, pred_train_phys, label_names, "train_phys")
metrics_val_phys   = regression_metrics(y_val_phys,   pred_val_phys,   label_names, "val_phys")

metrics_all_phys = pd.concat(
    [metrics_train_phys, metrics_val_phys],
    ignore_index=True
)

metrics_all_phys

## 2. Plot pred vs true

Qué mirar:

- Si los puntos siguen la diagonal.
- Si hay saturación en masas altas.
- Si hay regresión a la media.
- Si chi_eff está comprimido cerca de cero.

Mi predicción: chi_eff tendrá bastante regresión a la media.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_pred_vs_true(y_true, y_pred, label_names, split_name="test"):
    for j, label in enumerate(label_names):
        true = y_true[:, j]
        pred = y_pred[:, j]

        min_val = min(true.min(), pred.min())
        max_val = max(true.max(), pred.max())

        plt.figure(figsize=(8, 8))
        plt.scatter(true, pred, s=12, alpha=0.6)
        plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
        plt.xlabel(f"True {label}")
        plt.ylabel(f"Predicted {label}")
        plt.title(f"{split_name}: predicted vs true — {label}")
        plt.grid(True, alpha=0.3)
        plt.show()

In [ ]:
plot_pred_vs_true(
    y_true=y_val_phys,
    y_pred=pred_val_phys,
    label_names=label_names,
    split_name="val_phys"
)

## 3. Residuals per label

In [ ]:
def plot_residuals(y_true, y_pred, label_names, split_name="test"):
    residual = y_pred - y_true

    for j, label in enumerate(label_names):
        plt.figure(figsize=(6, 4))
        plt.hist(residual[:, j], bins=40, alpha=0.8)
        plt.axvline(0.0, linestyle="--")
        plt.xlabel(f"Residual: pred - true ({label})")
        plt.ylabel("Count")
        plt.title(f"{split_name}: residual distribution — {label}")
        plt.grid(True, alpha=0.3)
        plt.show()

In [ ]:
plot_residuals(y_val_phys, pred_val_phys, label_names, "val_phys")

In [ ]:
def plot_residual_vs_true(y_true, y_pred, label_names, split_name="test"):
    residual = y_pred - y_true

    for j, label in enumerate(label_names):
        plt.figure(figsize=(6, 4))
        plt.scatter(y_true[:, j], residual[:, j], s=12, alpha=0.6)
        plt.axhline(0.0, linestyle="--")
        plt.xlabel(f"True {label}")
        plt.ylabel(f"Residual: pred - true")
        plt.title(f"{split_name}: residual vs true — {label}")
        plt.grid(True, alpha=0.3)
        plt.show()

Esto es más importante que el histograma. Te dirá dónde falla el modelo.

In [ ]:
plot_residual_vs_true(y_val_phys, pred_val_phys, label_names, "val_phys")

## 4. Absolute error vs true value

Este plot es clave para Mondrian. Si el error aumenta con masa o depende de chi_eff, entonces tiene sentido usar bins condicionados.

In [ ]:
def plot_abs_error_vs_true(y_true, y_pred, label_names, split_name="test"):
    abs_error = np.abs(y_pred - y_true)

    for j, label in enumerate(label_names):
        plt.figure(figsize=(6, 4))
        plt.scatter(y_true[:, j], abs_error[:, j], s=12, alpha=0.6)
        plt.xlabel(f"True {label}")
        plt.ylabel(f"Absolute error")
        plt.title(f"{split_name}: absolute error vs true — {label}")
        plt.grid(True, alpha=0.3)
        plt.show()

In [ ]:
plot_abs_error_vs_true(y_val_phys, pred_val_phys, label_names, "val_phys")

## 5. Error vs SNR

In [ ]:
snr_val = network_snrs[val_idx]

In [ ]:
def plot_abs_error_vs_quantity(quantity, y_true, y_pred, label_names, quantity_name, split_name="val"):
    abs_error = np.abs(y_pred - y_true)

    for j, label in enumerate(label_names):
        plt.figure(figsize=(6, 4))
        plt.scatter(quantity, abs_error[:, j], s=12, alpha=0.6)
        plt.xlabel(quantity_name)
        plt.ylabel(f"Absolute error in {label}")
        plt.title(f"{split_name}: abs error vs {quantity_name} — {label}")
        plt.grid(True, alpha=0.3)
        plt.show()

In [ ]:
plot_abs_error_vs_quantity(
    quantity=snr_val,
    y_true=y_val_phys,
    y_pred=pred_val_phys,
    label_names=label_names,
    quantity_name="network SNR",
    split_name="val_phys"
)

For bins

In [ ]:
def metrics_by_quantity_bins(quantity, y_true, y_pred, label_names, bin_edges, quantity_name):
    rows = []

    for b in range(len(bin_edges) - 1):
        lo = bin_edges[b]
        hi = bin_edges[b + 1]

        mask = (quantity >= lo) & (quantity < hi)

        if mask.sum() == 0:
            continue

        residual = y_pred[mask] - y_true[mask]
        abs_error = np.abs(residual)

        for j, label in enumerate(label_names):
            rows.append({
                "quantity": quantity_name,
                "bin": f"[{lo:.2f}, {hi:.2f})",
                "count": int(mask.sum()),
                "label": label,
                "MAE": abs_error[:, j].mean(),
                "RMSE": np.sqrt((residual[:, j] ** 2).mean()),
                "bias": residual[:, j].mean(),
                "median_abs_error": np.median(abs_error[:, j]),
            })

    return pd.DataFrame(rows)

In [ ]:


snr_edges = np.linspace(10, 25, 7)

metrics_snr_val = metrics_by_quantity_bins(
    quantity=snr_val,
    y_true=y_val_phys,
    y_pred=pred_val_phys,
    label_names=label_names,
    bin_edges=snr_edges,
    quantity_name="network_snr"
)


snr_by_label = metrics_snr_val.sort_values(by="label", kind="stable")


display(snr_by_label)

## 6. Save predictions and embeddings

In [ ]:
output_path = RESULTS_DIR / f"{checkpoint_id}_predictions_embeddings.npz"

np.savez(
    output_path,

    pred_train=pred_train,
    pred_val=pred_val,
    pred_cal=pred_cal,
    pred_test=pred_test,

    y_train=y_train,
    y_val=y_val,
    y_cal=y_cal,
    y_test=y_test,

    pred_train_phys=pred_train_phys,
    pred_val_phys=pred_val_phys,
    pred_cal_phys=pred_cal_phys,
    pred_test_phys=pred_test_phys,

    y_train_phys=y_train_phys,
    y_val_phys=y_val_phys,
    y_cal_phys=y_cal_phys,
    y_test_phys=y_test_phys,

    emb_train=emb_train,
    emb_val=emb_val,
    emb_cal=emb_cal,
    emb_test=emb_test,

    idx_train=train_idx,
    idx_val=val_idx,
    idx_cal=cal_idx,
    idx_test=test_idx,

    y_mean=y_mean,
    y_std=y_std,

    label_names=np.array(label_names),

    best_epoch=checkpoint["epoch"],
    best_val_loss=checkpoint["best_val_loss"],
    checkpoint_path=str(checkpoint_path),
)

print("Saved:", output_path)